# LeafCare AI: training notebook

Trains the crop-disease classifier used by the LeafCare AI app (EfficientNetB0, PlantVillage, 38 classes)
and exports everything the app needs.

**How to run (Google Colab)**
1. *Runtime → Change runtime type →* **T4 GPU**.
2. *Runtime → Run all*. The first cell installs the exact TensorFlow/Keras versions used by the app and
   **restarts the runtime once**; after the restart, choose *Runtime → Run all* again.
3. Allow Google Drive access when asked. Checkpoints are saved to `MyDrive/leafcare-ai/checkpoints`, so if
   Colab disconnects, just *Run all* again and training resumes where it stopped.
4. At the end, `leafcare_export.zip` downloads automatically (a copy is also saved to Drive). Unzip it into
   the project folder (see the last cell).

Expected time on a T4: roughly 45–75 minutes. Set `QUICK_RUN = True` in the config cell for a
5-minute end-to-end check on a small subset first.

## 1. Setup: pin library versions

In [ ]:
# These MUST match requirements.txt, otherwise the saved model may not load in the app.
TF_VERSION = "2.21.0"
KERAS_VERSION = "3.15.1"

import importlib.metadata
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


if IN_COLAB and (installed_version("tensorflow") != TF_VERSION or installed_version("keras") != KERAS_VERSION):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tensorflow[and-cuda]=={TF_VERSION}", f"keras=={KERAS_VERSION}"],
                   check=True)
    print("Installed the pinned versions. The runtime will now restart:")
    print("when it reconnects, choose Runtime > Run all again.")
    os.kill(os.getpid(), 9)  # restarts the Colab runtime so the new versions are used

In [ ]:
import json
import re
import shutil
import subprocess
from pathlib import Path

import cv2
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from keras import layers
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
assert tf.__version__ == TF_VERSION and keras.__version__ == KERAS_VERSION, \
    "Library versions differ from requirements.txt - re-run the first cell."
print("GPU:", tf.config.list_physical_devices("GPU") or "none! (Runtime > Change runtime type > T4 GPU)")

## 2. Configuration

In [ ]:
QUICK_RUN = False        # True = small subset + 1 epoch per phase, for a fast end-to-end test

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_PHASE1 = 1 if QUICK_RUN else 5     # frozen backbone
EPOCHS_PHASE2 = 1 if QUICK_RUN else 10    # fine-tuning
LR_PHASE1, LR_PHASE2 = 1e-3, 1e-5
UNFREEZE_FRACTION = 0.30                  # top 30% of the backbone is fine-tuned in phase 2

keras.utils.set_random_seed(SEED)

# Everything the app needs is written under OUT and zipped at the end.
OUT = Path("/content/leafcare-ai") if IN_COLAB else Path("leafcare_export")
MODELS_DIR = OUT / "models"
FIG_DIR = OUT / "reports" / "figures"
SAMPLES_DIR = OUT / "app" / "assets" / "sample_images"
CACHE_DIR = Path("/content/tf_cache") if IN_COLAB else Path("tf_cache")   # fast local disk
for folder in (MODELS_DIR, FIG_DIR, SAMPLES_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# Checkpoints go to Google Drive so a Colab disconnect doesn't lose progress.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = Path("/content/drive/MyDrive/leafcare-ai/checkpoints")
else:
    CKPT_DIR = Path("checkpoints")
if QUICK_RUN:
    CKPT_DIR = CKPT_DIR / "quick_run"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoints:", CKPT_DIR)

## 3. Shared preprocessing

This cell is an **exact copy** of `preprocess` in `app/utils/preprocessing.py`, so training and the app prepare
images identically (`tests/test_preprocessing.py` checks this). EfficientNet rescales pixels internally,
so images stay in the 0–255 range: no division by 255 here.

In [ ]:
# --- shared preprocessing (must be identical in app/utils/preprocessing.py and notebooks/train_model.ipynb) ---
def preprocess(image):
    """Resize an RGB image tensor (H, W, 3) to the model input: float32, 224x224x3, values 0-255."""
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    return tf.cast(image, tf.float32)
# --- end shared preprocessing ---

## 4. Load PlantVillage and make a stratified 80 / 10 / 10 split

The first run downloads the PlantVillage colour images (~1 GB, a few minutes) from the dataset authors' GitHub
repository ([spMohanty/PlantVillage-Dataset](https://github.com/spMohanty/PlantVillage-Dataset)).
TensorFlow Datasets' copy isn't used because its download server (Mendeley Data) blocks Colab with HTTP 403.

In [ ]:
DATA_ROOT = Path("/content/PlantVillage-Dataset") if IN_COLAB else Path("PlantVillage-Dataset")
DATA_DIR = DATA_ROOT / "raw" / "color"
DONE_MARKER = DATA_ROOT / ".download_complete"   # guards against a half-finished download

if not DONE_MARKER.exists():
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/spMohanty/PlantVillage-Dataset.git", str(DATA_ROOT)], check=True)
    subprocess.run(["git", "-C", str(DATA_ROOT), "sparse-checkout", "set", "raw/color"], check=True)
    DONE_MARKER.touch()

# The ordered class list: index i = output neuron i of the model.
# Same names and order as models/class_names.json and the keys of data/diseases.json.
CLASS_NAMES = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Blueberry___healthy", "Cherry___healthy", "Cherry___Powdery_mildew",
    "Corn___Cercospora_leaf_spot Gray_leaf_spot", "Corn___Common_rust", "Corn___healthy",
    "Corn___Northern_Leaf_Blight", "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___healthy",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot", "Peach___healthy", "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
    "Potato___Early_blight", "Potato___healthy", "Potato___Late_blight", "Raspberry___healthy",
    "Soybean___healthy", "Squash___Powdery_mildew", "Strawberry___healthy", "Strawberry___Leaf_scorch",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___healthy", "Tomato___Late_blight",
    "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot", "Tomato___Tomato_mosaic_virus", "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
]
NUM_CLASSES = len(CLASS_NAMES)
(MODELS_DIR / "class_names.json").write_text(json.dumps(CLASS_NAMES, indent=2))


def folder_to_class(folder_name):
    """GitHub folder name -> app class name, e.g. 'Corn_(maize)___Common_rust_' -> 'Corn___Common_rust'."""
    crop, _, disease = folder_name.partition("___")
    crop = re.sub(r"_\(.*\)$", "", crop)
    return f"{crop}___{disease.rstrip('_')}"


folders = {folder_to_class(p.name): p for p in DATA_DIR.iterdir() if p.is_dir()}
assert set(folders) == set(CLASS_NAMES), f"Folder/class mismatch: {set(folders) ^ set(CLASS_NAMES)}"

paths, labels = [], []
for index, name in enumerate(CLASS_NAMES):
    files = sorted(p for p in folders[name].iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    paths += [str(p) for p in files]
    labels += [index] * len(files)
paths, labels = np.array(paths), np.array(labels)
print(f"{len(paths):,} images, {NUM_CLASSES} classes")

indices = np.arange(len(labels))
if QUICK_RUN:
    indices, _ = train_test_split(indices, train_size=0.15, stratify=labels, random_state=SEED)
train_idx, rest_idx = train_test_split(indices, test_size=0.2, stratify=labels[indices], random_state=SEED)
val_idx, test_idx = train_test_split(rest_idx, test_size=0.5, stratify=labels[rest_idx], random_state=SEED)
print(f"train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_idx):,}")

pd.Series(labels).map(lambda i: CLASS_NAMES[i]).value_counts().sort_values().plot.barh(
    figsize=(8, 10), color="#2F6B3A", title="Images per class")
plt.tight_layout()
plt.show()

## 5. Input pipelines (tf.data) and augmentation

Each split is cached on local disk after the first pass, then shuffled, preprocessed, batched and prefetched.
Augmentation (training only) makes the model more robust to real-world photos.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
TRAIN, VAL, TEST = 0, 1, 2
split_indices = {TRAIN: train_idx, VAL: val_idx, TEST: test_idx}

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.2, value_range=(0, 255)),
], name="augmentation")


def read_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return image, label


def random_crop(image):
    """Slight random crop (85-100% of the side), resized back to the model input size."""
    size = tf.cast(tf.random.uniform([], 0.85, 1.0) * IMG_SIZE, tf.int32)
    image = tf.image.random_crop(image, tf.stack([size, size, 3]))
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE))


def make_dataset(split, training=False):
    idx = split_indices[split]
    if training:
        # Files are listed class by class, so mix them once up front; the shuffle buffer below
        # alone would give batches dominated by one or two classes.
        idx = np.random.default_rng(SEED).permutation(idx)
    ds = (tf.data.Dataset.from_tensor_slices((paths[idx], labels[idx]))
          .map(read_image, num_parallel_calls=AUTOTUNE)
          .cache(str(CACHE_DIR / f"files_split{split}_{'quick' if QUICK_RUN else 'full'}")))
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda image, label: (preprocess(image), label), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda image, label: (random_crop(image), label), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda images, labels: (augment(images, training=True), labels), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(TRAIN, training=True)
val_ds = make_dataset(VAL)
test_ds = make_dataset(TEST)

# Preview some augmented training images.
images, batch_labels = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(np.clip(images[i].numpy(), 0, 255).astype("uint8"))
    plt.title(CLASS_NAMES[batch_labels[i]].replace("___", "\n"), fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Model: EfficientNetB0 + new classification head

Built as one **flat** functional model (the backbone is created on our input tensor instead of being nested),
so the last conv layer `top_conv` is directly accessible for Grad-CAM in the app.

In [ ]:
def build_model(num_classes):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
    backbone = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_tensor=inputs)
    x = layers.GlobalAveragePooling2D(name="avg_pool")(backbone.output)
    x = layers.Dropout(0.3, name="head_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="leafcare_efficientnetb0")


def backbone_layers(model):
    """All EfficientNet layers: everything between the input and the new head."""
    names = [layer.name for layer in model.layers]
    return model.layers[1:names.index("avg_pool")]


def compile_model(model, learning_rate):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])


def training_callbacks(phase):
    return [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(str(CKPT_DIR / f"{phase}_best.keras"),
                                        monitor="val_accuracy", save_best_only=True),
        keras.callbacks.BackupAndRestore(str(CKPT_DIR / f"{phase}_backup")),  # resume after a disconnect
    ]


def as_floats(history):
    return {key: [float(v) for v in values] for key, values in history.items()}


def trainable_count(model):
    return sum(int(np.prod(w.shape)) for w in model.trainable_weights)

## 7. Phase 1: train the head (backbone frozen)

In [ ]:
phase1_history_file = CKPT_DIR / "phase1_history.json"
if phase1_history_file.exists():
    model = keras.models.load_model(CKPT_DIR / "phase1_best.keras")
    history1 = json.loads(phase1_history_file.read_text())
    print("Phase 1 already finished: loaded the best phase-1 model from Drive.")
else:
    model = build_model(NUM_CLASSES)
    for layer in backbone_layers(model):
        layer.trainable = False
    compile_model(model, LR_PHASE1)
    print(f"Trainable parameters: {trainable_count(model):,}")
    fit = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE1, callbacks=training_callbacks("phase1"))
    history1 = as_floats(fit.history)
    phase1_history_file.write_text(json.dumps(history1))

## 8. Phase 2: fine-tune the top 30% of the backbone

BatchNormalization layers stay frozen (their statistics are kept from ImageNet), and a small learning rate
avoids destroying the pre-trained features.

In [ ]:
phase2_history_file = CKPT_DIR / "phase2_history.json"
if phase2_history_file.exists():
    model = keras.models.load_model(CKPT_DIR / "phase2_best.keras")
    history2 = json.loads(phase2_history_file.read_text())
    print("Phase 2 already finished: loaded the best fine-tuned model from Drive.")
else:
    backbone = backbone_layers(model)
    first_unfrozen = int(len(backbone) * (1 - UNFREEZE_FRACTION))
    for i, layer in enumerate(backbone):
        layer.trainable = i >= first_unfrozen and not isinstance(layer, layers.BatchNormalization)
    compile_model(model, LR_PHASE2)
    print(f"Unfroze backbone layers from '{backbone[first_unfrozen].name}' onward; "
          f"trainable parameters: {trainable_count(model):,}")
    fit = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE2, callbacks=training_callbacks("phase2"))
    history2 = as_floats(fit.history)
    phase2_history_file.write_text(json.dumps(history2))

## 9. Evaluation: curves, test metrics, confusion matrix

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
GREEN, AMBER = "#2F6B3A", "#E0A33B"


def pretty(class_name):
    crop, _, disease = class_name.partition("___")
    return f"{crop.replace('_', ' ')}: {disease.replace('_', ' ')}"


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    train_values = history1[metric] + history2[metric]
    val_values = history1[f"val_{metric}"] + history2[f"val_{metric}"]
    epochs = range(1, len(train_values) + 1)
    ax.plot(epochs, train_values, "o-", color=GREEN, label="train")
    ax.plot(epochs, val_values, "o-", color=AMBER, label="validation")
    ax.axvline(len(history1[metric]) + 0.5, color="grey", linestyle="--", linewidth=1)
    ax.text(len(history1[metric]) + 0.6, ax.get_ylim()[1], " fine-tuning", va="top", color="grey")
    ax.set(title=title, xlabel="epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
y_true = np.concatenate([batch_labels.numpy() for _, batch_labels in test_ds])
y_prob = model.predict(test_ds, verbose=1)
y_pred = y_prob.argmax(axis=1)

test_accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Test accuracy: {test_accuracy:.4f}   Macro F1: {macro_f1:.4f}   (target: accuracy >= 0.95)")

report = classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASS_NAMES,
                               output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_df.to_csv(FIG_DIR / "classification_report.csv")
report_df.head(NUM_CLASSES).sort_values("f1-score").head(10)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES), normalize="true")
plt.figure(figsize=(18, 15))
sns.heatmap(cm, cmap="Greens", vmin=0, vmax=1, square=True, linewidths=0.3, linecolor="#EEE",
            xticklabels=[pretty(c) for c in CLASS_NAMES], yticklabels=[pretty(c) for c in CLASS_NAMES],
            cbar_kws={"label": "fraction of true class", "shrink": 0.6})
plt.xlabel("Predicted class", fontsize=12)
plt.ylabel("True class", fontsize=12)
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)
plt.title("Normalised confusion matrix (test set)", fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix.png", bbox_inches="tight")
plt.show()

## 10. Sample predictions: correct and incorrect

In [ ]:
def get_test_images(wanted):
    """Return {test-set position: image (224x224x3, 0-255)} for the requested positions."""
    wanted, found = set(int(i) for i in wanted), {}
    for position, (image, _) in enumerate(test_ds.unbatch()):
        if position in wanted:
            found[position] = image.numpy()
            if len(found) == len(wanted):
                break
    return found


correct = np.flatnonzero(y_pred == y_true)
wrong = np.flatnonzero(y_pred != y_true)
rng = np.random.default_rng(SEED)
shown = list(rng.choice(correct, size=min(8, len(correct)), replace=False)) + \
        list(rng.choice(wrong, size=min(8, len(wrong)), replace=False))
images = get_test_images(shown)

cols = 4
rows = int(np.ceil(len(shown) / cols))
plt.figure(figsize=(16, 4.3 * rows))
for n, position in enumerate(shown):
    ok = y_pred[position] == y_true[position]
    plt.subplot(rows, cols, n + 1)
    plt.imshow(images[position].astype("uint8"))
    plt.title(f"True: {pretty(CLASS_NAMES[y_true[position]])}\n"
              f"Pred: {pretty(CLASS_NAMES[y_pred[position]])} ({y_prob[position].max():.0%})",
              fontsize=8, color=GREEN if ok else "#C2412D")
    plt.axis("off")
plt.suptitle("Sample test predictions (green = correct, red = wrong)", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "sample_predictions.png", bbox_inches="tight")
plt.show()

## 11. Grad-CAM examples

Same algorithm as `app/utils/gradcam.py`: the gradient of the predicted class with respect to the `top_conv`
feature maps shows which regions drove the decision.

In [ ]:
gradcam_model = keras.Model(model.inputs, [model.get_layer("top_conv").output, model.output])


def gradcam_heatmap(image, class_index):
    with tf.GradientTape() as tape:
        conv_maps, preds = gradcam_model(image[np.newaxis, ...], training=False)
        score = preds[:, class_index]
    grads = tape.gradient(score, conv_maps)
    heatmap = tf.nn.relu(tf.reduce_sum(conv_maps[0] * tf.reduce_mean(grads, axis=(0, 1, 2)), axis=-1)).numpy()
    return heatmap / heatmap.max() if heatmap.max() > 0 else heatmap


def overlay(image, heatmap, alpha=0.45):
    rgb = image.astype("uint8")
    heat = np.uint8(255 * cv2.resize(heatmap, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_CUBIC).clip(0, 1))
    colored = cv2.cvtColor(cv2.applyColorMap(heat, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(colored, alpha, rgb, 1 - alpha, 0)


# One correctly classified test image for each of 8 classes spread across the class list.
chosen_classes = np.linspace(0, NUM_CLASSES - 1, 8).astype(int)
examples = {c: correct[y_true[correct] == c][0] for c in chosen_classes if np.any(y_true[correct] == c)}
images = get_test_images(examples.values())

plt.figure(figsize=(16, 4.2 * int(np.ceil(len(examples) / 2))))
for n, (class_index, position) in enumerate(examples.items()):
    image = images[position]
    heatmap = gradcam_heatmap(tf.constant(image), int(class_index))
    for k, (panel, label) in enumerate([(image.astype("uint8"), "photo"), (overlay(image, heatmap), "Grad-CAM")]):
        plt.subplot(int(np.ceil(len(examples) / 2)), 4, 2 * n + k + 1)
        plt.imshow(panel)
        plt.title(f"{pretty(CLASS_NAMES[class_index])}\n{label}", fontsize=8)
        plt.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "gradcam_examples.png", bbox_inches="tight")
plt.show()

## 12. Export: model, metrics, sample images, zip

In [ ]:
metrics = {
    "test_accuracy": float(test_accuracy),
    "macro_f1": float(macro_f1),
    "num_test_images": int(len(y_true)),
    "num_classes": NUM_CLASSES,
    "per_class_f1": {name: float(report[name]["f1-score"]) for name in CLASS_NAMES},
    "tf_version": tf.__version__,
    "keras_version": keras.__version__,
}
(MODELS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

model_path = MODELS_DIR / "leaf_model.keras"
model.save(model_path)

# Sanity check: the saved file reloads and gives the same predictions.
reloaded = keras.models.load_model(model_path, compile=False)
check_batch = next(iter(test_ds))[0][:8]
assert np.allclose(model.predict(check_batch, verbose=0), reloaded.predict(check_batch, verbose=0), atol=1e-4)
print(f"Saved {model_path} ({model_path.stat().st_size / 1e6:.1f} MB) and verified it reloads.")

# Real sample images for the app's "Try a sample" row (correctly classified test images).
SAMPLE_CLASSES = ["Tomato___Early_blight", "Potato___Late_blight", "Apple___Apple_scab",
                  "Corn___Common_rust", "Grape___Black_rot", "Tomato___healthy"]
sample_positions = {name: correct[y_true[correct] == CLASS_NAMES.index(name)][0]
                    for name in SAMPLE_CLASSES
                    if name in CLASS_NAMES and np.any(y_true[correct] == CLASS_NAMES.index(name))}
sample_images = get_test_images(sample_positions.values())
for name, position in sample_positions.items():
    cv2.imwrite(str(SAMPLES_DIR / f"{name}.jpg"),
                cv2.cvtColor(sample_images[position].astype("uint8"), cv2.COLOR_RGB2BGR))
print("Sample images:", sorted(p.name for p in SAMPLES_DIR.iterdir()))

In [ ]:
# Reference for the app's "is this a leaf?" check: for each class, the average feature vector (avg_pool layer)
# of 25 training images, scaled to unit length. Same computation as scripts/build_leaf_centroids.py.
embedder = keras.Model(model.inputs, model.get_layer("avg_pool").output)
ref_idx = [train_idx[labels[train_idx] == c][:25] for c in range(NUM_CLASSES)]
ref_ds = (tf.data.Dataset.from_tensor_slices(paths[np.concatenate(ref_idx)])
          .map(lambda path: preprocess(read_image(path, 0)[0]), num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE))
features = embedder.predict(ref_ds, verbose=0)
features /= np.linalg.norm(features, axis=1, keepdims=True)
groups = np.split(features, np.cumsum([len(i) for i in ref_idx])[:-1])
centroids = np.stack([g.mean(0) / np.linalg.norm(g.mean(0)) for g in groups]).astype("float32")
np.save(MODELS_DIR / "leaf_centroids.npy", centroids)
print("Saved leaf_centroids.npy", centroids.shape)

In [ ]:
archive = shutil.make_archive(str(OUT.parent / "leafcare_export"), "zip", root_dir=OUT)
print("Created", archive)
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(OUT))

if IN_COLAB:
    shutil.copy(archive, CKPT_DIR.parent)  # backup copy in MyDrive/leafcare-ai/
    from google.colab import files
    files.download(archive)

## 13. Next steps (on your computer)

1. Delete the placeholder images in `app/assets/sample_images/` (`placeholder_*.jpg`).
2. Unzip `leafcare_export.zip` **into the project root**. It fills `models/`, `reports/figures/` and
   `app/assets/sample_images/`, replacing the placeholder model.
3. Run `python scripts/validate_knowledge_base.py` and `pytest`, then `streamlit run app/Home.py`.
4. Copy the test accuracy and macro F1 printed above into the README's Results table.